In [ ]:
## RAG Application Using TypeSense

In [10]:
import typesense
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
client = typesense.Client(
    {
        "nodes": [
            {"host": os.getenv("TYPESENSE_HOST"), "port": "443", "protocol": "https"}
        ],
        "api_key": os.getenv("TYPESENSE_API_KEY"),
        "connection_timeout_seconds": 2,
    }
)

books_schema = {
    "name": "books",
    "fields": [
        {"name": "title", "type": "string"},
        {"name": "authors", "type": "string[]", "facet": True},
        {"name": "publication_year", "type": "int32", "facet": True},
        {"name": "ratings_count", "type": "int32"},
        {"name": "average_rating", "type": "float"},
    ],
    "default_sorting_field": "ratings_count",
}

print(client.collections.create(books_schema))

{'created_at': 1779364522, 'curation_sets': [], 'default_sorting_field': 'ratings_count', 'enable_nested_fields': False, 'fields': [{'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'title', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'authors', 'optional': False, 'sort': False, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'string[]'}, {'facet': True, 'index': True, 'infix': False, 'locale': '', 'name': 'publication_year', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': True, 'infix': False, 'locale': '', 'name': 'ratings_count', 'optional': False, 'sort': True, 'stem': False, 'stem_dictionary': '', 'store': True, 'truncate_len': 100, 'type': 'int32'}, {'facet': False, 'index': T

In [12]:
client

In [18]:
with open('books.jsonl','r', encoding='utf-8') as jsonl_files:
    data=jsonl_files.read()
    client.collections['books'].documents.import_(data)

In [ ]:
search_parameters = {
    "q": "harry potter",
    "query_by": "title,authors",
    "sort_by": "ratings_count:desc",
}

client.collections["books"].documents.search(search_parameters)

{'facet_counts': [],
 'found': 17,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}},
  {'document': {'authors': ['J.K. Rowling', ' Mary GrandPré', ' R

In [ ]:
search_parameters = {
    "q": "harry potter",
    "query_by": "title,authors",
    "filter_by": "publication_year:<1998",
    "sort_by": "publication_year:desc",
}

client.collections["books"].documents.search(search_parameters)

{'facet_counts': [],
 'found': 1,
 'hits': [{'document': {'authors': ['J.K. Rowling', ' Mary GrandPré'],
    'average_rating': 4.44,
    'id': '2',
    'image_url': 'https://images.gr-assets.com/books/1474154022m/3.jpg',
    'publication_year': 1997,
    'ratings_count': 4602479,
    'title': "Harry Potter and the Philosopher's Stone"},
   'highlight': {'title': {'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Harry', 'Potter'],
     'snippet': "<mark>Harry</mark> <mark>Potter</mark> and the Philosopher's Stone"}],
   'text_match': 1157451471441102969,
   'text_match_info': {'best_field_score': '2211897868289',
    'best_field_weight': 15,
    'fields_matched': 1,
    'num_tokens_dropped': 0,
    'score': '1157451471441102969',
    'tokens_matched': 2,
    'typo_prefix_score': 0}}],
 'out_of': 9979,
 'page': 1,
 'request_params': {'collection_name

In [28]:
search_parameters = {
    "q": "experiment",
    "query_by": "title",
    "facet_by": "authors",
    "sort_by": "average_rating:desc",
}

client.collections["books"].documents.search(search_parameters)

{'facet_counts': [{'counts': [{'count': 1,
     'highlighted': ' Käthe Mazur',
     'value': ' Käthe Mazur'},
    {'count': 1, 'highlighted': 'Mahatma Gandhi', 'value': 'Mahatma Gandhi'},
    {'count': 1, 'highlighted': 'Gretchen Rubin', 'value': 'Gretchen Rubin'},
    {'count': 1,
     'highlighted': 'James Patterson',
     'value': 'James Patterson'}],
   'field_name': 'authors',
   'sampled': False,
   'stats': {'total_values': 4}}],
 'found': 3,
 'hits': [{'document': {'authors': ['James Patterson'],
    'average_rating': 4.08,
    'id': '569',
    'image_url': 'https://images.gr-assets.com/books/1339277875m/13152.jpg',
    'publication_year': 2005,
    'ratings_count': 172302,
    'title': 'The Angel Experiment'},
   'highlight': {'title': {'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}},
   'highlights': [{'field': 'title',
     'matched_tokens': ['Experiment'],
     'snippet': 'The Angel <mark>Experiment</mark>'}],
   'text_match': 5787301

In [30]:
### LangChain + Typesense + GroqLLM + RAG Application

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Typesense
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_groq import ChatGroq

In [31]:

loader = TextLoader('data/text_files/ai_essay.txt')
document = loader.load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
docs = text_splitter.split_documents(document)

embeddings = HuggingFaceEmbeddings()


/var/folders/13/xrrj5z411yj45kh1jd_ds8v80000gn/T/ipykernel_28950/910004201.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings()
/var/folders/13/xrrj5z411yj45kh1jd_ds8v80000gn/T/ipykernel_28950/910004201.py:6: LangChainDeprecationWarning: Default values for HuggingFaceEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceEmbeddings constructor instead.
  embeddings = HuggingFaceEmbeddings()
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9876.07it/s]


In [ ]:
docsearch = Typesense.from_documents(
    docs,
    embeddings,
    typesense_client_params={
        "host": os.getenv("TYPESENSE_HOST"),
        "port": "443",
        "protocol": "https",
        "typesense_api_key": os.getenv("TYPESENSE_API_KEY"),
        "typesense_collection_name": "lang-chain",
    },
)

In [33]:
query='what is ai?'
found_docs = docsearch.similarity_search(query)
print(found_docs[0].page_content)

The Rise of Artificial Intelligence: A Comprehensive Journey Through Modern AI

I remember the first time I truly understood what artificial intelligence could do. It was late 2022, and I was playing around with a chatbot that could actually hold a conversation. Not the clunky, robotic responses I was used to from customer service bots, but something that felt almost human. That moment changed how I thought about technology forever.

Artificial intelligence isn't just about robots taking over the world like we see in movies. It's much more subtle and, honestly, more interesting than that. AI is already woven into our daily lives in ways most people don't even notice. When Netflix recommends a show you end up binging all weekend, that's AI. When your phone unlocks just by looking at it, that's AI too. When Gmail finishes your sentences before you do, yep, also AI.

The Foundations: Machine Learning Basics


In [34]:
## Retriever 
retriever = docsearch.as_retriever()
retriever

VectorStoreRetriever(tags=['Typesense', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.typesense.Typesense object at 0x16007e120>, search_kwargs={})

In [ ]:
query = 'what is ai?'
retriever.invoke(query)[0].page_content

Document(metadata={'source': 'data/text_files/ai_essay.txt'}, page_content="The Rise of Artificial Intelligence: A Comprehensive Journey Through Modern AI\n\nI remember the first time I truly understood what artificial intelligence could do. It was late 2022, and I was playing around with a chatbot that could actually hold a conversation. Not the clunky, robotic responses I was used to from customer service bots, but something that felt almost human. That moment changed how I thought about technology forever.\n\nArtificial intelligence isn't just about robots taking over the world like we see in movies. It's much more subtle and, honestly, more interesting than that. AI is already woven into our daily lives in ways most people don't even notice. When Netflix recommends a show you end up binging all weekend, that's AI. When your phone unlocks just by looking at it, that's AI too. When Gmail finishes your sentences before you do, yep, also AI.\n\nThe Foundations: Machine Learning Basics"